In [1]:
%reload_ext dotenv
%dotenv

import warnings
import logging
import datetime
import os
import json

from faster_whisper import WhisperModel
from utils.utils import mp42m4a, cut_blanks, audio_paths, format_duration
from pydub import AudioSegment

logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)
warnings.filterwarnings(action="ignore")   # <--- ignore after imports

cannot find .env file


In [2]:
ASR_MODEL = "deepdml/faster-whisper-large-v3-turbo-ct2"
MIN_SILENCE_LEN = 800
MANUAL_SILENCE_LEN = 300
SILENCE_THRESH = -60
# ASR_MODEL = "XA9/Belle-faster-whisper-large-v3-zh-punct"

WHISPER_MODEL = WhisperModel(ASR_MODEL, device="cuda", compute_type="float16")

In [3]:
def mfa(audio_file_dir):
    os.system(
        f"mfa align --output_format json \
                --use_threading \
                --use_mp \
                --overwrite \
                --clean \
                --final_clean \
                {audio_file_dir}/chunks \
                mandarin_china_mfa \
                mandarin_mfa \
                {audio_file_dir}/chunks"
    )

In [4]:
def replace_special_chars(
    text,
):  # remove space and "! " if the first letter is space or "! "
    # Check if the text starts with "!" or " " and ends with " "
    if text.startswith("! ") or text.startswith(" "):
        # Replace the special characters with an empty string
        text = text.replace("!", "").replace(
            " ", "", 1
        )  # Only replace the first occurrence
    text = text.replace(",", "，").replace("?", "？")
    return text


def format_to_srt(seconds):  # Convert seconds to SRT's timecode
    dt = datetime.datetime(1, 1, 1) + datetime.timedelta(seconds=seconds)
    formatted_time = "{:02d}:{:02d}:{:02d},{:03d}".format(
        dt.hour, dt.minute, dt.second, dt.microsecond // 1000
    )
    return formatted_time

In [5]:
def generate_subtitles(audio_file, sub_list, subtitle_format="srt"):
    if subtitle_format == "srt":
        srt_content = ""
        srt_number = 1
        for sub in sub_list:  # Add subtitle's index number
            sub_srt = f"{sub['start_time_str']} --> {sub['end_time_str']}\n{sub['text']}\n\n"
            srt_content += str(srt_number) + "\n" + sub_srt
            srt_number = srt_number + 1
        with open(f"{audio_file}.srt", "w", encoding="utf-8") as srt_file:
            srt_file.write(srt_content)
    elif subtitle_format == "json":
        with open(f"{audio_file}.json", "w", encoding="utf-8") as json_file:
            json_file.write(str(sub_list).replace("'", '"'))
    elif subtitle_format == "txt":
        with open(f"{audio_file}.txt", "w", encoding="utf-8") as txt_file:
            for sub in sub_list:
                txt_file.write(sub["text"])

In [6]:
def transcribe_audio(audio_file, subtitle_format="srt"):
    segments, info = WHISPER_MODEL.transcribe(
        f"{audio_file}.wav",
        word_timestamps=True,
        initial_prompt="以下是普通话的句子。",
        beam_size=5,
        language="zh",
        max_new_tokens=433,
        condition_on_previous_text=False,
        vad_filter=False,
        vad_parameters=dict(min_silence_duration_ms=500),
    )

    sub_list: list[dict[str, str]] = []
    for segment in segments:
        start_time_str = format_to_srt(segment.start)
        end_time_str = format_to_srt(segment.end)
        sub_text = replace_special_chars(segment.text)
        print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, sub_text))
        sub_entry = {
            "start_time_str": start_time_str,
            "end_time_str": end_time_str,
            "text": sub_text,
        }
        sub_list.append(sub_entry)  # Add formatted subtitles to list
    generate_subtitles(audio_file, sub_list, subtitle_format)
    print("")
    print("Saved: " + os.path.abspath(f"{audio_file}.{subtitle_format}"))

In [7]:
# Concatenate audio files with silence intervals and adjust timestamps
def concatenate_audio_and_adjust_timestamps(audio_file, chunks):
    audio_file_dir, audio_file_name = audio_paths(audio_file)
    combined_audio = AudioSegment.empty()
    combined_data = {}
    sequence = 0
    cumulative_time = 0.0

    for i in range(chunks):
        chunk_audio_file = f"{audio_file_dir}/chunks/{audio_file_name}_chunk{i}"
        wav_file_path = f"{chunk_audio_file}.wav"
        json_file_path = f"{chunk_audio_file}.json"
        
        if not os.path.exists(wav_file_path):
            print(f"Warning: {wav_file_path} not found, skipping...")
            continue
        
        # Load audio chunk
        audio_chunk = AudioSegment.from_file(wav_file_path, format="wav")
        
        # Add the audio chunk to combined audio
        combined_audio += audio_chunk
        
        # Process JSON if exists
        if os.path.exists(json_file_path):
            try:
                with open(json_file_path, 'r', encoding='utf-8') as f:
                    chunk_data = json.load(f)
                
                # MFA JSON structure: {'start': ..., 'end': ..., 'tiers': {'words': [...]}}
                if 'tiers' in chunk_data and 'words' in chunk_data['tiers']:
                    words_tier = chunk_data['tiers']['words']
                    if 'entries' in words_tier:
                        for entry in words_tier['entries']:
                            word_text = entry[2].strip()
                            if word_text and word_text != '':  # Skip empty intervals
                                combined_data[sequence] = {
                                    "word": word_text,
                                    "start_time": entry[0] + cumulative_time,
                                    "end_time": entry[1] + cumulative_time
                                }
                                sequence += 1
            except Exception as e:
                print(f"Error processing {json_file_path}: {e}")
        
        # Update cumulative time (chunk duration + silence)
        chunk_duration_sec = len(audio_chunk) / 1000.0
        cumulative_time += chunk_duration_sec
        
        # Add silence interval between chunks (except after the last chunk)
        if i < chunks - 1:
            silence = AudioSegment.silent(duration=MANUAL_SILENCE_LEN)
            combined_audio += silence
            cumulative_time += MANUAL_SILENCE_LEN / 1000.0

    # Save combined audio file
    combined_audio_file = f"{audio_file_dir}/{audio_file_name}_combined.m4a"
    combined_audio.export(combined_audio_file, format="ipod")
    combined_json_file = f"{audio_file_dir}/{audio_file_name}_combined.json"
    with open(combined_json_file, 'w', encoding='utf-8') as f:
        json.dump(combined_data, f, ensure_ascii=False, indent=2)

    original_audio = AudioSegment.from_file(f"{audio_file_dir}/{audio_file_name}.m4a", format="m4a")
    original_audio_duration = len(original_audio) / 1000.0
    print(f"Combined {sequence} word entries from {chunks} chunks")
    print(f"Original audio duration: {format_duration(original_audio_duration)}")
    print(f"Total audio duration: {format_duration(len(combined_audio) / 1000.0)}")
    print(f"Saved audio: {os.path.abspath(combined_audio_file)}")
    print(f"Saved JSON: {os.path.abspath(combined_json_file)}")

In [11]:
def align_audio_with_edited_json(original_json_file, edited_json_file, original_audio_file, output_audio_file):
    """
    Compare original and edited JSON files, then generate new audio file
    that only contains segments present in the edited JSON.
    
    Args:
        original_json_file: Path to the original JSON file from ASR/MFA
        edited_json_file: Path to the manually edited JSON file
        original_audio_file: Path to the original audio file
        output_audio_file: Path to save the aligned audio file
    """
    # Load both JSON files
    with open(original_json_file, 'r', encoding='utf-8') as f:
        original_data = {int(k): v for k, v in json.load(f).items()}
    
    with open(edited_json_file, 'r', encoding='utf-8') as f:
        edited_data = {int(k): v for k, v in json.load(f).items()}
    
    # Load original audio
    original_audio = AudioSegment.from_file(original_audio_file, format="m4a")
    print(f"Original audio duration: {format_duration(len(original_audio) / 1000.0)}")
    end_ms = len(original_audio)
    rm_sequence = 0
    
    # Iterate through edited data in order
    for seq_id in reversed(sorted(original_data.keys())):
        if seq_id not in edited_data:
            # Get original timing
            original_entry = original_data.get(str(seq_id), original_data.get(seq_id))
            
            # Case 1: If the previous segment is also removed, extend the end time only
            if seq_id + 1 in edited_data:
                end_ms = int(original_entry['end_time'] * 1000)
            if seq_id - 1 not in edited_data:
                print(f"Removed sequence {seq_id}", original_entry)
                rm_sequence += 1
                continue
            start_ms = int(original_entry['start_time'] * 1000)
            print(f"Removed sequence {seq_id}", original_entry, start_ms, end_ms)

            # Combine the two parts to create the new audio segment without the chunk
            original_audio = original_audio[:start_ms] + original_audio[end_ms:]
            rm_sequence += 1
    
    # Export new audio file
    original_audio.export(output_audio_file, format="ipod")
    
    print(f"Original {len(original_data)} segments, removed {rm_sequence} unused segments, {len(original_data) - rm_sequence} segments remaining")
    print(f"New audio duration: {format_duration(len(original_audio) / 1000.0)}")
    print(f"Saved audio: {os.path.abspath(output_audio_file)}")

In [ ]:
audio_file = "../youtube/guardiola/控制国家的盎撒老爷有多真？在新英格兰起码不假【美国大选地理01】"
audio_file_dir, audio_file_name = audio_paths(audio_file)
mp42m4a(audio_file)
chunks = cut_blanks(audio_file, MIN_SILENCE_LEN, SILENCE_THRESH)
for i in range(chunks):
    chunk_audio_file = f"{audio_file_dir}/chunks/{audio_file_name}_chunk{i}"
    transcribe_audio(chunk_audio_file, subtitle_format="txt")
mfa(audio_file_dir)
concatenate_audio_and_adjust_timestamps(audio_file, chunks)

In [12]:
align_audio_with_edited_json(
    f"{audio_file_dir}/{audio_file_name}_combined.json",
    f"{audio_file_dir}/{audio_file_name}_edited.json",
    f"{audio_file_dir}/{audio_file_name}_combined.m4a",
    f"{audio_file_dir}/{audio_file_name}_edited.m4a",
)

Original audio duration: 34:51:000
Removed sequence 234 {'word': '说', 'start_time': 78.95999908447266, 'end_time': 79.20999908447266} 78959 79209
Removed sequence 215 {'word': '种', 'start_time': 71.44000244140625, 'end_time': 71.61000061035156}
Removed sequence 214 {'word': '一', 'start_time': 71.36000061035156, 'end_time': 71.44000244140625} 71360 71610
Removed sequence 103 {'word': '的', 'start_time': 33.86000061035156, 'end_time': 33.91999816894531}
Removed sequence 102 {'word': '政治', 'start_time': 33.619998931884766, 'end_time': 33.86000061035156}
Removed sequence 101 {'word': '美国', 'start_time': 33.38999938964844, 'end_time': 33.619998931884766}
Removed sequence 100 {'word': '尤其', 'start_time': 33.15999984741211, 'end_time': 33.38999938964844} 33159 33919
Removed sequence 93 {'word': '种', 'start_time': 30.010000228881836, 'end_time': 30.15999984741211}
Removed sequence 92 {'word': '一', 'start_time': 29.959999084472656, 'end_time': 30.010000228881836}
Removed sequence 91 {'word': '的'